# Graduate-Level Characterization of the 532 nm and 1064 nm Lasers

This project analyzes the supplied measurements as three distinct experiments:

1. **Spatial:** Coherent COMPASS 315M-100, user-reported 130 mW CW output at 532 nm, directly intercepted by a Basler acA3088-57uc color CMOS camera.
2. **Temporal:** Thorlabs GSL106A, externally triggered at an intended 200 kHz and measured with a legacy Thorlabs PDA36A and a Keysight/Agilent DSO6104A.
3. **Spectral:** both sources measured with a HORIBA iHR550 and Synapse Plus detector using 300, 1200, and 1800 groove/mm gratings.

The project distinguishes four levels of conclusion: **supported**, **supported with caveat**, **provisional**, and **not supported**. A fit result is never promoted to a physical claim merely because an optimizer converged.

In [ ]:
from pathlib import Path
import json, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown


def _locate_project_root(start=None):
    """Find the extracted project folder even if Jupyter was launched one directory above it."""
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for base in candidates:
        if (base / "laser_characterization.py").is_file() and (base / "outputs").is_dir():
            return base.resolve()
        try:
            children = sorted(base.glob("Laser_Characterization_Project*")) + sorted(base.glob("laser_characterization_project*"))
        except OSError:
            children = []
        for child in children:
            if child.is_dir() and (child / "laser_characterization.py").is_file() and (child / "outputs").is_dir():
                return child.resolve()
    raise FileNotFoundError(
        "Could not locate the laser-characterization project folder. Keep the notebooks, "
        "laser_characterization.py, laser_characterization_core.py, and outputs/ together."
    )

PROJECT_ROOT = _locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from laser_characterization import *

OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)
with (PROJECT_ROOT / "experiment_metadata.json").open() as handle:
    EXPERIMENT = json.load(handle)

# Raw data are optional for viewing the complete precomputed analysis.
# If present, they enable reconstruction of raw-data plots.
try:
    DATA_ROOT = find_data_root(PROJECT_ROOT)
except FileNotFoundError:
    DATA_ROOT = None
    for candidate in [
        PROJECT_ROOT / "data" / "532 and 1064",
        PROJECT_ROOT.parent / "532 and 1064",
        PROJECT_ROOT.parent / "data" / "532 and 1064",
    ]:
        if candidate.is_dir():
            DATA_ROOT = candidate.resolve()
            break
if DATA_ROOT is None:
    warnings.warn(
        "Raw data folder not found. This is OK: all precomputed result tables are included, "
        "and raw-data-only plots will be skipped."
    )


def source_path(stored_path):
    """Resolve a portable path stored relative to the `532 and 1064` data root."""
    p = Path(str(stored_path))
    if p.is_absolute() and p.exists():
        return p
    if DATA_ROOT is None:
        return p
    # Current V4 tables store paths relative to DATA_ROOT.
    q = Path(DATA_ROOT) / p
    if q.exists():
        return q
    # Backward compatibility for older absolute paths.
    parts = p.parts
    if "532 and 1064" in parts:
        i = parts.index("532 and 1064")
        q = Path(DATA_ROOT) / Path(*parts[i+1:])
        if q.exists():
            return q
    return q

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.25,
})
print("Project:", PROJECT_ROOT)
print("Raw data:", DATA_ROOT if DATA_ROOT else "not mounted (using included precomputed results)")


## Corrected instrument interpretation

The iHR550 designation refers to a **550 mm focal length** spectrometer. Its entrance aperture is **f/6.4**; 6.4 is not a focal length. The legacy PDA36A is also kept distinct from the newer PDA36A2. At the recorded 0 dB gain setting, the analysis uses the legacy manual values of 10 MHz nominal bandwidth, 0.750 kV/A into 50 Ω, and 1.51 kV/A into a high-impedance load.

In [ ]:
known = pd.DataFrame([
    ["532 nm source", "Coherent COMPASS 315M-100", "user supplied"],
    ["532 nm operation", "CW; >130 mW measured with PM100D/S121C", "user supplied; not treated as a manufacturer guarantee"],
    ["1064 nm source", "Thorlabs GSL106A", "user supplied / filename consistency"],
    ["GSL trigger", "external; intended 200 kHz", "user supplied; measured from CH3 in notebook 02"],
    ["Spatial camera", "Basler acA3088-57uc, BayerRG12, direct interception", "user supplied"],
    ["Focusing optic", "Thorlabs 100 mm plano-convex", "user supplied; exact part/orientation unknown"],
    ["Temporal detector", "legacy PDA36A, 0 dB", "user supplied"],
    ["Spectrometer", "HORIBA iHR550: 550 mm focal length, f/6.4", "manufacturer specification"],
    ["Spectral calibration", "two dominant visible Hg lines, all three gratings", "user supplied; residuals not recorded"],
], columns=["item", "value", "provenance"])
display(known)

## Data inventory and reproducibility outputs

The native camera data, processed legacy PNG scans, oscilloscope exports, and HORIBA spectra are intentionally kept separate. The legacy PNG files are never mixed numerically with the native 12-bit RAW files.

In [ ]:
inventory = pd.read_csv(OUTPUT_DIR / "data_inventory.csv")
inventory["total_MB"] = inventory["total_bytes"] / 1e6
display(inventory[["family", "count", "total_MB", "example"]].round({"total_MB": 2}))

outputs = sorted(p.name for p in OUTPUT_DIR.glob("*"))
print(f"Precomputed output files: {len(outputs)}")
print("\n".join(outputs))

## What the uncertainty columns mean

- A standard deviation across three camera frames is **repeatability at one z position**, not a complete beam-diameter uncertainty.
- A covariance-derived fit error is **conditional on the chosen model and data**, and excludes pixel-scale error, z-position error, background-model error, saturation, and lobe-selection uncertainty.
- Manufacturer specifications are listed as instrument context. They are not combined into a formal expanded uncertainty because the required calibration and correlation information is absent.
- CSV files retain more digits than should normally appear in prose so the calculations are reproducible. The interpretation cells use sensible rounding and do not treat displayed digits as accuracy.

In [ ]:
with (OUTPUT_DIR / "integrated_results.json").open() as handle:
    summary = json.load(handle)

headline = pd.DataFrame([
    ["Spatial", "Reportable M²", summary["spatial"]["reportable_m2"], summary["spatial"]["reason"]],
    ["Temporal", "Measured repetition rate", True, f'{summary["temporal"]["measured_repetition_rate_hz_range_long_records"][0]:.6f} to {summary["temporal"]["measured_repetition_rate_hz_range_long_records"][1]:.6f} Hz'],
    ["Temporal", "Optical pulse widths", False, "PDA36A-limited or detector-convolved electrical waveforms"],
    ["Spectral", "Intrinsic narrow linewidth", False, "instrument response not removed; many peaks also undersampled"],
    ["Spectral", "Broad picosecond-mode bandwidth", True, f'{summary["spectral"]["picosecond_fwhm_nm_range"][0]:.3f} to {summary["spectral"]["picosecond_fwhm_nm_range"][1]:.3f} nm'],
], columns=["domain", "claim", "supported", "reason/result"])
display(headline)

## Recommended workflow

Run `01_Spatial_532nm.ipynb`, `02_Temporal_1064nm.ipynb`, and `03_Spectral_532_1064nm.ipynb` before `04_Integrated_Summary.ipynb`. The precomputed outputs were generated by:

```bash
python run_all_analyses.py --force
```

Set average-power values in `outputs/average_power_inputs.csv` to enable pulse-energy calculations. Do not enter guessed responsivity or ND transmission values; the notebook deliberately blocks absolute peak-power reconstruction without them.